# Can Sparse Mixture-of-Experts Modularity Mitigate Catastrophic Forgetting in Class-Incremental Learning?

Companion notebook for the paper by **Arthur Estefanato Lopes and Filipe Mutz** (Universidade Federal do Espírito Santo). It reproduces every figure and table in the paper, plus the additional analyses summarized in its discussion.

**Summary.** We compare a sparse Mixture-of-Experts (SMoE) with a dense MLP under class-incremental learning on MNIST and CIFAR-10. Modularity alone gives no retention advantage; freezing experts trades plasticity for retention, and its plasticity cost comes from old classes winning the final decision rather than from a failure to learn. The router specializes experts around visual primitives, but the shared input projection keeps rotating the latent space, so routing never stabilizes. An oracle router that fixes one expert per class is the only variant that surpasses the dense baseline.

## How to use this notebook

Every experiment stores its raw results in `results/` and its figures in `figures/`. With `RETRAIN = False` (default), experiments whose results are already in `results/` are **loaded instead of retrained**, so the whole notebook runs in a few minutes and redraws every figure and table. Set `RETRAIN = True` to retrain everything from scratch.

Runs are deterministic: the same seed produces the same numbers across sessions (Section 1).

## Map from the paper to this notebook

| Paper | Notebook section | Retraining time (CPU, approx.) |
|---|---|---|
| Figure 1b (protocol) | 7 | seconds |
| Figure 2 (information loss and plasticity) | 5.1 | ~4 h (MNIST and CIFAR-10, 8 seeds) |
| Figures 3, 4, 5 (routing) | 5.2 | ~40 min |
| Table 1 (task- vs class-incremental) | 5.3 | ~1 h |
| Table 2 (three tasks) and the control split | 5.4 | ~4 h |
| Section 4.5 (input projection) | 5.5 | ~2 h |
| Additional analyses (discussion) | 5.6 | ~1.5 h |

Times are rough estimates for a CPU runtime; a GPU is considerably faster.

## 1. Setup

Determinism matters here. During the project, some SMoE configurations produced different results across GPU sessions under the same seed: the router combined expert outputs through indexed accumulation, which has no deterministic GPU implementation. The model below combines expert outputs densely instead (mathematically identical), and PyTorch is told to refuse any non-deterministic operation.

In [ ]:
import os
# must be set before importing torch: cuBLAS needs it to run deterministically on GPU
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json, math, random
from dataclasses import dataclass, replace, asdict
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, confusion_matrix

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RETRAIN = False                  # True: retrain every experiment instead of loading results/
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"device: {device} | deterministic: {torch.are_deterministic_algorithms_enabled()} | retrain: {RETRAIN}")

## 2. Configuration

Every experiment is a variation of one configuration object. The defaults below are the paper's reference model: an SMoE with 10 experts, top-2 routing, balancing weight $\alpha = 10$, output masking and synaptic freezing (the **SMoE (freeze)** configuration). Each experiment changes only the fields it studies, through `cfg.with_(field=value)`.

The task configurations define which classes form each task:

- `SEQUENTIAL_MNIST`: digits 0–4, then 5–9 (the main comparison);
- `CIFAR_RANDOM`: the ten CIFAR-10 classes assigned randomly to two groups of five;
- `MORPHOLOGICAL_MNIST`: three tasks grouped by stroke shape: curved (3, 5, 8), circular (6, 9, 0) and straight (1, 2, 4, 7). The digit 2, whose curved top and straight base fit either group, is placed so that all ten digits are covered;
- `CONTROL_MNIST`: three tasks with the same sizes (3, 3, 4) but in numerical order, which mixes shapes within each task. It is the control for the morphological configuration.

In [ ]:
def random_split(n_groups=2, n_classes=10, seed=0):
    # Assigns the classes randomly to n_groups groups of equal size.
    classes = list(range(n_classes))
    random.Random(seed).shuffle(classes)
    size = n_classes // n_groups
    return tuple(tuple(sorted(classes[i * size:(i + 1) * size])) for i in range(n_groups))


SEQUENTIAL_MNIST = ((0, 1, 2, 3, 4), (5, 6, 7, 8, 9))
CIFAR_RANDOM = random_split(n_groups=2, n_classes=10, seed=0)      # ((1,3,5,7,8), (0,2,4,6,9))
MORPHOLOGICAL_MNIST = ((3, 5, 8), (6, 9, 0), (1, 4, 7, 2))
CONTROL_MNIST = ((0, 1, 2), (3, 4, 5), (6, 7, 8, 9))

CIFAR_CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
                     "dog", "frog", "horse", "ship", "truck"]


@dataclass(frozen=True)
class Config:
    # --- components under study ---
    use_input_projection: bool = True     # shared linear projection before the MoE layer
    output_masking: bool = True           # mask the logits of classes outside the current task
    mask_first_task: Optional[bool] = None  # None: same masking in every task (see Section 4)
    freeze_experts: bool = True           # synaptic freezing at task transitions
    reserve_policy: str = "minimum"       # experts kept free: "minimum" (one) or "proportional"
    routing: str = "learned"              # "learned" or "oracle" (one dedicated expert per class)
    top_k: int = 2
    n_experts: int = 10
    alpha: float = 10.0                   # weight of the load-balancing loss
    model: str = "smoe"                   # "smoe" or "mlp"
    # --- architecture and training ---
    d_model: int = 128                    # width of the projected representation
    expert_hidden: Optional[int] = None   # expert hidden width; None = twice its input width
    lr: float = 0.002
    batch_size: int = 64
    epochs_per_phase: int = 5
    # --- protocol ---
    dataset: str = "mnist"                # "mnist" or "cifar10"
    tasks: Tuple[Tuple[int, ...], ...] = SEQUENTIAL_MNIST
    split_first_task: bool = True         # train the first task in two disjoint halves
    seeds: Tuple[int, ...] = (42, 123, 2024, 7, 99, 555, 831, 2718)

    @property
    def n_tasks(self):
        return len(self.tasks)

    def with_(self, **kwargs):
        return replace(self, **kwargs)


BASE = Config()
CIFAR = BASE.with_(dataset="cifar10", tasks=CIFAR_RANDOM)
DIAGNOSIS_SEEDS = (42, 123, 2024)          # seeds of the evaluation diagnosis (Section 5.3)
print(BASE)

## 3. Data

Images are standardized per channel and flattened: the models have no convolutional layers. This is adequate for MNIST, but on CIFAR-10 it discards spatial structure and caps absolute accuracy for every configuration, so CIFAR-10 results should be read as comparisons between conditions.

In [ ]:
_TRANSFORMS = {
    "mnist": transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]),
    "cifar10": transforms.Compose([transforms.ToTensor(),
                                   transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))]),
}
_DATASETS = {}

def load_dataset(name):
    # Returns (train set, test set, flattened input size), cached after the first call.
    if name not in _DATASETS:
        cls = {"mnist": datasets.MNIST, "cifar10": datasets.CIFAR10}[name]
        train = cls("./data", train=True, download=True, transform=_TRANSFORMS[name])
        test = cls("./data", train=False, download=True, transform=_TRANSFORMS[name])
        _DATASETS[name] = (train, test, 28 * 28 if name == "mnist" else 32 * 32 * 3)
    return _DATASETS[name]


def _targets(dataset):
    t = dataset.targets
    return t.tolist() if torch.is_tensor(t) else list(t)


def class_indices(dataset, classes):
    classes = set(int(c) for c in classes)
    return [i for i, y in enumerate(_targets(dataset)) if int(y) in classes]


def make_loader(dataset, classes, batch_size=64, shuffle=True):
    return DataLoader(Subset(dataset, class_indices(dataset, classes)),
                      batch_size=batch_size, shuffle=shuffle)


def make_half_loaders(dataset, classes, batch_size=64, seed=42):
    # Splits the training data of a task into two disjoint halves.
    idx = class_indices(dataset, classes)
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(idx), generator=g).tolist()
    shuffled = [idx[i] for i in perm]
    half = len(shuffled) // 2
    return (DataLoader(Subset(dataset, shuffled[:half]), batch_size=batch_size, shuffle=True),
            DataLoader(Subset(dataset, shuffled[half:]), batch_size=batch_size, shuffle=True))


_TEST_LOADERS = {}

def test_loaders(dataset_name, tasks):
    # One test loader per task (fixed order, cached).
    key = (dataset_name, tasks)
    if key not in _TEST_LOADERS:
        _, test, _ = load_dataset(dataset_name)
        _TEST_LOADERS[key] = [make_loader(test, list(c), batch_size=500, shuffle=False) for c in tasks]
    return _TEST_LOADERS[key]


def n_output_classes(tasks):
    return max(c for task in tasks for c in task) + 1

## 4. Models, training and evaluation

**Dense baseline.** An MLP with two hidden layers of 128 units and ReLU.

**SMoE.** The flattened image is projected to 128 dimensions by a learned linear layer (the *input projection*). A router scores each expert by cosine similarity with a learned vector, and a softmax with learned temperature turns the scores into probabilities. The top-$k$ experts ($k = 2$) process each input, weighted by their probabilities from the full softmax, without renormalizing over the selected experts. Each expert is a two-layer network with GELU and a hidden layer twice as wide as its input. A cosine classifier maps the output to classes.

**Load balancing.** Over the set $F$ of experts not frozen, with $\bar P_i$ the batch-mean routing probability of expert $i$ renormalized over $F$, the loss adds $\alpha\,|F|\sum_{i\in F}\bar P_i^2$, which is minimal when usage is uniform over $F$.

**Synaptic freezing.** At each task transition, experts whose mean routing probability over the finished task exceeds $0.4/n$ have their weights and router vectors frozen, and at least one expert is always kept free. In practice this threshold is not selective: the balancing loss drives every expert's usage toward $1/n > 0.4/n$, so all but one expert end up frozen.

**Output masking.** The logits of classes outside the current task are set to a large negative value before the loss, so the gradient reaches only the current task's output units. It is applied in *every* task, including the first: masking only from the second task onward biases results toward old classes, because the first task then pushes down the logits of future classes while later tasks never push down those of earlier ones (Section 5.6 measures this).

**Oracle routing (ideal router).** An upper bound for routing quality: each class gets a dedicated expert, created when the class first appears. During training each sample goes to its class's expert, while the router learns, through an auxiliary cross-entropy term, to predict that assignment; at test time the router alone selects the expert (top-1).

**Protocol and metrics.** Models are trained on the first task in two disjoint halves (probing expert reuse), then on each subsequent task, with the optimizer state preserved across phases. After each task, the macro F1-score is measured on the test set of every task. **Retention** is the F1-score on earlier tasks after the last task; **plasticity** is the F1-score on the newest task; **information loss** is 100% minus retention.

In [ ]:
class DenseMLP(nn.Module):
    def __init__(self, input_dim, n_classes, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_classes))

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))


class CosineLinear(nn.Module):
    # Cosine similarity between inputs and learned row vectors, scaled by a learned sigma.
    # Rows can be frozen: their gradient is zeroed by a hook.
    def __init__(self, in_features, out_features, sigma=10.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.sigma = nn.Parameter(torch.tensor(sigma))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.register_buffer("frozen_rows", torch.zeros(out_features, dtype=torch.bool))
        self.weight.register_hook(self._zero_frozen_rows)

    def _zero_frozen_rows(self, grad):
        if self.frozen_rows.any():
            grad = grad.clone()
            grad[self.frozen_rows] = 0
        return grad

    def freeze_rows(self, rows):
        for r in rows:
            if r < self.frozen_rows.size(0):
                self.frozen_rows[r] = True

    def forward(self, x):
        return F.linear(F.normalize(x, p=2, dim=-1), F.normalize(self.weight, p=2, dim=-1)) * self.sigma


def expand_cosine_linear(old, n_new_rows, device):
    # Larger CosineLinear keeping the learned rows (used when the oracle adds classes).
    new = CosineLinear(old.weight.shape[1], old.weight.shape[0] + n_new_rows).to(device)
    with torch.no_grad():
        new.weight[:old.weight.shape[0]] = old.weight
        new.sigma.copy_(old.sigma)
    return new


class Expert(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.GELU(), nn.Linear(d_hidden, d_in))

    def forward(self, x):
        return self.net(x)


class SMoELayer(nn.Module):
    def __init__(self, d_in, n_experts, top_k, expert_hidden=None):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = min(top_k, n_experts)
        self.router = CosineLinear(d_in, n_experts)
        hidden = expert_hidden if expert_hidden is not None else d_in * 2
        self.experts = nn.ModuleList([Expert(d_in, hidden) for _ in range(n_experts)])
        self.register_buffer("free", torch.ones(n_experts, dtype=torch.bool))

    def forward(self, x):
        probs = F.softmax(self.router(x), dim=-1)
        top_p, top_i = torch.topk(probs, self.top_k, dim=-1)
        weights = torch.zeros(x.size(0), self.n_experts, device=x.device, dtype=x.dtype)
        weights.scatter_(1, top_i, top_p)
        # dense combination: identical to routing each sample to its top-k experts,
        # but without indexed accumulation, which has no deterministic GPU implementation
        out = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            out = out + expert(x) * weights[:, i:i + 1]
        n_free = self.free.sum().float()
        if n_free > 1:
            p = probs[:, self.free]
            p = p / (p.sum(dim=-1, keepdim=True) + 1e-9)
            l_bal = n_free * torch.sum(p.mean(dim=0) ** 2)
        else:
            l_bal = torch.tensor(0.0, device=x.device)
        return out, l_bal, probs

    def freeze_experts(self, ids):
        self.router.freeze_rows(ids)
        for e in ids:
            if e < len(self.experts):
                for p in self.experts[e].parameters():
                    p.requires_grad = False
                self.free[e] = False


class SMoE(nn.Module):
    # SMoE with learned routing; the input projection is optional.
    def __init__(self, input_dim, n_classes, cfg):
        super().__init__()
        self.use_input_projection = cfg.use_input_projection
        d = cfg.d_model if cfg.use_input_projection else input_dim
        self.input_projection = nn.Linear(input_dim, cfg.d_model) if cfg.use_input_projection else nn.Identity()
        self.moe = SMoELayer(d, cfg.n_experts, cfg.top_k, expert_hidden=cfg.expert_hidden)
        self.classifier = CosineLinear(d, n_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = F.relu(self.input_projection(x)) if self.use_input_projection else x
        out, l_bal, probs = self.moe(h)
        return self.classifier(out), l_bal, probs

    def freeze_experts(self, ids):
        self.moe.freeze_experts(ids)


class OracleSMoE(nn.Module):
    # Ideal router: one dedicated expert per class, added when the class first appears.
    def __init__(self, input_dim, cfg, device):
        super().__init__()
        self.use_input_projection = cfg.use_input_projection
        self.d = cfg.d_model if cfg.use_input_projection else input_dim
        self.input_projection = (nn.Linear(input_dim, cfg.d_model).to(device)
                                 if cfg.use_input_projection else nn.Identity().to(device))
        self.experts = nn.ModuleList().to(device)
        self.router = None
        self.classifier = None
        self.class_to_expert = {}
        self.device = device
        self.expert_hidden = cfg.expert_hidden

    def add_classes(self, new_classes):
        hidden = self.expert_hidden if self.expert_hidden is not None else self.d * 2
        for c in sorted(new_classes):
            if c in self.class_to_expert:
                continue
            self.class_to_expert[c] = len(self.experts)
            self.experts.append(Expert(self.d, hidden).to(self.device))
        n = len(self.experts)
        if self.router is None:
            self.router = CosineLinear(self.d, n).to(self.device)
        elif self.router.weight.shape[0] < n:
            self.router = expand_cosine_linear(self.router, n - self.router.weight.shape[0], self.device)
        n_classes = max(self.class_to_expert) + 1
        if self.classifier is None:
            self.classifier = CosineLinear(self.d, n_classes).to(self.device)
        elif self.classifier.weight.shape[0] < n_classes:
            self.classifier = expand_cosine_linear(
                self.classifier, n_classes - self.classifier.weight.shape[0], self.device)

    def forward(self, x, targets=None):
        x = x.view(x.size(0), -1)
        h = F.relu(self.input_projection(x)) if self.use_input_projection else x
        router_logits = self.router(h)
        out = torch.zeros_like(h)
        if targets is not None:          # training: route by the true class
            for c in torch.unique(targets).tolist():
                m = targets == c
                out[m] = self.experts[self.class_to_expert[c]](h[m])
        else:                            # test: the router decides alone (top-1)
            chosen = torch.argmax(router_logits, dim=-1)
            for e in torch.unique(chosen).tolist():
                m = chosen == e
                out[m] = self.experts[e](h[m])
        return self.classifier(out), router_logits


def count_parameters(cfg, n_classes=10):
    # Parameter count of an SMoE configuration, without instantiating it.
    input_dim = 3072 if cfg.dataset == "cifar10" else 784
    d = cfg.d_model if cfg.use_input_projection else input_dim
    hidden = cfg.expert_hidden if cfg.expert_hidden is not None else d * 2
    total = (input_dim * cfg.d_model + cfg.d_model) if cfg.use_input_projection else 0
    total += cfg.n_experts * ((d * hidden + hidden) + (hidden * d + d))
    total += cfg.n_experts * d + 1 + n_classes * d + 1
    return total


def equalized_expert_hidden(cfg_reference, cfg_target, tolerance=0.02):
    # Smallest expert hidden width that gives cfg_target (about) the parameter count of
    # cfg_reference. Without it, removing the input projection would make each expert
    # operate on the raw image and multiply the parameter count (32x on MNIST).
    goal = count_parameters(cfg_reference)
    best_h, best_diff = None, float("inf")
    for h in range(1, 4096):
        p = count_parameters(cfg_target.with_(expert_hidden=h))
        if abs(p - goal) < best_diff:
            best_h, best_diff = h, abs(p - goal)
        if abs(p - goal) / goal <= tolerance:
            return h
    return best_h

In [ ]:
def train_phase(model, loader, active_classes, cfg, optimizer, mask, is_moe):
    # One training phase. Returns the routing probabilities of the last epoch (SMoE only),
    # which decide which experts get frozen at the next transition.
    criterion = nn.CrossEntropyLoss()
    model.train()
    last_probs = []
    for _ in range(cfg.epochs_per_phase):
        epoch_probs = []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            if is_moe:
                logits, l_bal, probs = model(x)
                epoch_probs.append(probs.detach().cpu())
            else:
                logits, l_bal = model(x), 0.0
            if mask:
                inactive = torch.ones(logits.size(1), dtype=torch.bool, device=device)
                inactive[active_classes] = False
                logits = logits.clone()
                logits[:, inactive] = -1e9
            loss = criterion(logits, y)
            if is_moe:
                loss = loss + cfg.alpha * l_bal
            loss.backward()
            optimizer.step()
        if is_moe and epoch_probs:
            last_probs = epoch_probs
    return torch.cat(last_probs, dim=0) if last_probs else None


def train_phase_oracle(model, loader, active_classes, cfg, optimizer, mask):
    # Training with oracle routing. The router's auxiliary loss is masked like the
    # classification loss; otherwise every step would push down the router rows of
    # earlier classes.
    criterion = nn.CrossEntropyLoss()
    model.train()
    for _ in range(cfg.epochs_per_phase):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, router_logits = model(x, targets=y)
            if mask:
                inactive = torch.ones(logits.size(1), dtype=torch.bool, device=device)
                inactive[active_classes] = False
                logits = logits.clone()
                logits[:, inactive] = -1e9
                active_experts = [model.class_to_expert[c] for c in active_classes]
                inactive_r = torch.ones(router_logits.size(1), dtype=torch.bool, device=device)
                inactive_r[active_experts] = False
                router_logits = router_logits.clone()
                router_logits[:, inactive_r] = -1e9
            router_targets = torch.tensor([model.class_to_expert[t.item()] for t in y], device=device)
            loss = criterion(logits, y) + criterion(router_logits, router_targets)
            loss.backward()
            optimizer.step()


def evaluate_f1(model, loader, classes, mode):
    # Macro F1-score over the classes of one task, with an unrestricted argmax over all
    # outputs (class-incremental evaluation).
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            if mode == "oracle":
                out = model(x, targets=None)[0]
            elif mode == "smoe":
                out = model(x)[0]
            else:
                out = model(x)
            preds.extend(torch.max(out.data, 1)[1].cpu().numpy())
            targets.extend(y.numpy())
    return f1_score(targets, preds, labels=list(classes), average="macro", zero_division=0) * 100


def reserve_size(cfg, remaining_tasks):
    # Experts kept free at a transition: one, or a share proportional to the remaining tasks.
    if cfg.reserve_policy == "proportional":
        return max(1, int(round(cfg.n_experts * remaining_tasks / cfg.n_tasks)))
    return 1


def select_experts_to_freeze(mean_prob, n_experts, free=None, rel_threshold=0.4, min_reserve=1):
    threshold = rel_threshold / n_experts
    candidates = torch.where(mean_prob > threshold)[0].tolist()
    if free is not None:
        candidates = [i for i in candidates if bool(free[i])]
        n_free = int(free.sum().item())
    else:
        n_free = n_experts
    max_frozen = max(0, n_free - min_reserve)
    if len(candidates) > max_frozen:
        candidates = sorted(candidates, key=lambda i: mean_prob[i].item(), reverse=True)[:max_frozen]
    return candidates


def run_seed(cfg, seed):
    # Runs the full task sequence for one seed. Returns the F1 matrix:
    # matrix[i][j] = F1 on task j after training up to task i.
    torch.manual_seed(seed)
    train_set, _, input_dim = load_dataset(cfg.dataset)
    tasks = [list(t) for t in cfg.tasks]
    n_classes = n_output_classes(cfg.tasks)
    test = test_loaders(cfg.dataset, cfg.tasks)

    oracle = cfg.model == "smoe" and cfg.routing == "oracle"
    is_moe = cfg.model == "smoe"
    mode = "oracle" if oracle else ("smoe" if is_moe else "mlp")
    mask_first = cfg.output_masking if cfg.mask_first_task is None else cfg.mask_first_task

    if oracle:
        model = OracleSMoE(input_dim, cfg, device)
        model.add_classes(tasks[0])
    elif is_moe:
        model = SMoE(input_dim, n_classes, cfg).to(device)
    else:
        model = DenseMLP(input_dim, n_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)

    matrix = [[None] * len(tasks) for _ in tasks]
    frozen = []
    for i, classes in enumerate(tasks):
        if i == 0 and cfg.split_first_task:
            phases = list(make_half_loaders(train_set, classes, batch_size=cfg.batch_size, seed=seed))
        else:
            phases = [make_loader(train_set, classes, batch_size=cfg.batch_size)]
        mask = mask_first if i == 0 else cfg.output_masking
        probs = None
        for loader in phases:
            if oracle:
                train_phase_oracle(model, loader, classes, cfg, optimizer, mask=mask)
            else:
                probs = train_phase(model, loader, classes, cfg, optimizer, mask=mask, is_moe=is_moe)
        for j, classes_j in enumerate(tasks):
            matrix[i][j] = evaluate_f1(model, test[j], classes_j, mode)
        if i < len(tasks) - 1:
            if oracle:
                model.add_classes(tasks[i + 1])
                optimizer = optim.Adam(model.parameters(), lr=cfg.lr)   # new router/classifier
                frozen.append(len(tasks[i]))
            elif is_moe and cfg.freeze_experts and probs is not None:
                ids = select_experts_to_freeze(probs.mean(dim=0), cfg.n_experts, free=model.moe.free,
                                               min_reserve=reserve_size(cfg, len(tasks) - (i + 1)))
                if ids:
                    model.freeze_experts(ids)
                frozen.append(len(ids))
            else:
                frozen.append(0)
    return {"matrix": matrix, "frozen": frozen}


def run_config(cfg, label):
    runs = [run_seed(cfg, s) for s in cfg.seeds]
    return {"label": label, "config": _serializable(asdict(cfg)), "seeds": list(cfg.seeds), "runs": runs}


def _serializable(obj):
    if isinstance(obj, (tuple, list)):
        return [_serializable(o) for o in obj]
    if isinstance(obj, dict):
        return {k: _serializable(v) for k, v in obj.items()}
    return obj


def cached(name, compute):
    # Loads results/<name>.json if present (and RETRAIN is False); otherwise computes and saves it.
    path = RESULTS_DIR / f"{name}.json"
    if path.exists() and not RETRAIN:
        print(f"[loaded {path}]")
        return json.load(open(path))
    out = compute()
    json.dump(out, open(path, "w"))
    print(f"[saved {path}]")
    return out


def summarize(result):
    # Mean and standard deviation over seeds of the metrics used in the paper.
    M = np.array([r["matrix"] for r in result["runs"]], dtype=float)    # (seeds, i, j)
    last = M.shape[1] - 1
    s = {"label": result["label"], "matrix_mean": M.mean(0), "matrix_std": M.std(0)}
    retention = M[:, last, :last].mean(axis=1)
    s["retention"], s["retention_std"] = retention.mean(), retention.std()
    s["plasticity"], s["plasticity_std"] = M[:, last, last].mean(), M[:, last, last].std()
    s["first_task"], s["first_task_std"] = M[:, 0, 0].mean(), M[:, 0, 0].std()
    rel = np.array([np.mean([M[k, last, j] / M[k, j, j] * 100 for j in range(last) if M[k, j, j] > 1e-6] or [0])
                    for k in range(M.shape[0])])
    s["relative_retention"], s["relative_retention_std"] = rel.mean(), rel.std()
    return s


def print_summary(results, title):
    print(f"\n=== {title} ===")
    print(f"{'configuration':<30} {'Task 1 learned':>16} {'retention':>16} {'plasticity':>16} {'rel. retention':>16}")
    for r in results:
        s = summarize(r)
        print(f"{s['label']:<30} {s['first_task']:>8.2f} ± {s['first_task_std']:<5.2f} "
              f"{s['retention']:>8.2f} ± {s['retention_std']:<5.2f} "
              f"{s['plasticity']:>8.2f} ± {s['plasticity_std']:<5.2f} "
              f"{s['relative_retention']:>8.2f} ± {s['relative_retention_std']:<5.2f}")


def print_matrix(result):
    s = summarize(result)
    n = s["matrix_mean"].shape[0]
    print(f"\nF1 matrix, {s['label']} (row: after training task i; column: task j)")
    print(" " * 14 + "".join(f"{'T' + str(j + 1):>16}" for j in range(n)))
    for i in range(n):
        print(f"after T{i + 1}".ljust(14) + "".join(
            f"{s['matrix_mean'][i][j]:>9.2f} ± {s['matrix_std'][i][j]:<5.2f}" for j in range(n)))

## 5. Experiments

### 5.1 Does modularity contribute to knowledge retention? (Figure 2)

Six configurations on MNIST (digits 0–4, then 5–9) and on CIFAR-10 (random groups of five classes), eight seeds each:

| Configuration | Output masking | Synaptic freezing | Routing |
|---|---|---|---|
| MLP | no | — | — |
| SMoE | no | no | learned |
| MLP (masked) | yes | — | — |
| SMoE (no freeze) | yes | no | learned |
| SMoE (freeze) | yes | yes | learned |
| SMoE (ideal router) | yes | — | oracle |

The comparison that isolates modularity is **MLP (masked)** against **SMoE (no freeze)**: both use the same output masking and no other protection, so any difference comes from the modular architecture itself.

In [ ]:
def main_configurations(base):
    return [("MLP",                 base.with_(model="mlp", output_masking=False)),
            ("SMoE",                base.with_(output_masking=False, freeze_experts=False)),
            ("MLP (masked)",        base.with_(model="mlp", output_masking=True)),
            ("SMoE (no freeze)",    base.with_(output_masking=True, freeze_experts=False)),
            ("SMoE (freeze)",       base.with_(output_masking=True, freeze_experts=True)),
            ("SMoE (ideal router)", base.with_(routing="oracle"))]

main_mnist = cached("main_mnist", lambda: [run_config(c, l) for l, c in main_configurations(BASE)])
main_cifar = cached("main_cifar10", lambda: [run_config(c, l) for l, c in main_configurations(CIFAR)])

print_summary(main_mnist, "MNIST, two tasks")
print_summary(main_cifar, "CIFAR-10, two tasks")

**Reading the results.** Without masking, both architectures forget Task 1 entirely. On MNIST, the unfrozen SMoE retains *less* than the masked MLP under the same masking, so modularity alone does not help; freezing matches the masked MLP's retention at a large plasticity cost; only the ideal router exceeds the dense baseline. On CIFAR-10 the pattern repeats, with the modularity-only comparison ending in a tie. Masking lowers the MLP's first-task score on CIFAR-10 but not the SMoE's, which is why the relative retention (the share of what each model had learned that it still retains) is reported alongside the absolute one.

Figure 2 shows information loss (100% minus retention, so that forgetting appears as a bar) and plasticity.

In [ ]:
LABELS = ["MLP", "SMoE", "MLP (masked)", "SMoE (no freeze)", "SMoE (freeze)", "SMoE (ideal router)"]
COLORS = ["#999999", "#555555", "#E69F00", "#56B4E9", "#009E73", "#CC79A7"]    # Okabe-Ito
TEXT_WIDTH = 5.9    # inches: the paper's text width; figures are drawn at their printed size
FIG_RC = {"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 10.5,
          "axes.labelsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10}


def draw_figure_2(mnist, cifar, path=FIGURES_DIR / "f1_phases_combined.png"):
    by_label = {"MNIST": {r["label"]: summarize(r) for r in mnist},
                "CIFAR-10": {r["label"]: summarize(r) for r in cifar}}
    panels = [("MNIST", "Information loss", "retention"), ("MNIST", "Plasticity (T2)", "plasticity"),
              ("CIFAR-10", "Information loss", "retention"), ("CIFAR-10", "Plasticity (T2)", "plasticity")]
    with plt.rc_context(FIG_RC):
        fig, axes = plt.subplots(1, 4, figsize=(TEXT_WIDTH, 2.6), sharey=True)
        x = np.arange(len(LABELS))
        for ax, (ds, title, key) in zip(axes, panels):
            s = [by_label[ds][l] for l in LABELS]
            vals = [100 - v[key] if key == "retention" else v[key] for v in s]
            ax.bar(x, vals, yerr=[v[key + "_std"] for v in s], color=COLORS, width=0.78, capsize=2,
                   error_kw={"linewidth": 0.9, "ecolor": "#222"}, zorder=3)
            ax.set_title(f"{ds}\n{title}", fontweight="bold", pad=4)
            ax.set_xticks([]); ax.set_ylim(0, 105); ax.set_yticks([0, 25, 50, 75, 100])
            ax.grid(axis="y", linestyle=":", alpha=0.7, zorder=0)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
        axes[0].set_ylabel("Macro F1-score (%)")
        fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS], LABELS, loc="lower center",
                   ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02), handlelength=1.2, columnspacing=1.2)
        plt.tight_layout(rect=(0, 0.15, 1, 1), w_pad=1.3)
        plt.savefig(path, dpi=300, facecolor="white")
        plt.show()

draw_figure_2(main_mnist, main_cifar)

### 5.2 Does routing remain stable across tasks? (Figures 3, 4 and 5)

Three analyses on MNIST, each with one seed:

- **Allocation (Figure 3):** per-class routing probability and expert usage after each half of Task 1 and after Task 2 (10 experts, with freezing). All panels share one color scale, so small variations are not exaggerated.
- **Specialization (Figure 4):** an SMoE with 20 experts is trained on all ten classes at once, and each test image is assigned to its top-1 expert. The figure shows the first test images of the five most used experts; the printed table gives, for every expert, the share of each class among the test images it receives.
- **Revisitation (Figure 5):** models alternate between the two tasks for 8 cycles (1 epoch per phase, no freezing), and the routing matrix is extracted after cycles 1, 2, 4 and 8. If routing converged, the matrices would stop changing.

The routing matrices and sample indices are saved in `results/`, so the figures can be redrawn without retraining.

In [ ]:
def routing_matrix(model, dataset, classes, cfg, max_batches=20):
    # Mean routing probability per (class, expert), over the first batches of the data.
    loader = make_loader(dataset, classes, batch_size=256, shuffle=False)
    total, count = np.zeros((10, cfg.n_experts)), np.zeros(10)
    model.eval()
    with torch.no_grad():
        for b, (x, y) in enumerate(loader):
            if b >= max_batches:
                break
            probs = model(x.to(device))[2].cpu().numpy()
            for c in np.unique(y.numpy()):
                m = y.numpy() == c
                total[c] += probs[m].sum(axis=0)
                count[c] += m.sum()
    valid = count > 0
    matrix = np.zeros((10, cfg.n_experts))
    matrix[valid] = total[valid] / count[valid, None]
    return matrix, valid


def allocation_snapshots(cfg, seed=42):
    cfg = cfg.with_(routing="learned", model="smoe")
    train_set, _, input_dim = load_dataset(cfg.dataset)
    tasks = [list(t) for t in cfg.tasks]
    torch.manual_seed(seed)
    model = SMoE(input_dim, n_output_classes(cfg.tasks), cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    half1, half2 = make_half_loaders(train_set, tasks[0], batch_size=cfg.batch_size, seed=seed)
    snaps = []
    probs = train_phase(model, half1, tasks[0], cfg, opt, mask=cfg.output_masking, is_moe=True)
    snaps.append(("After half 1", *routing_matrix(model, train_set, tasks[0], cfg)))
    probs = train_phase(model, half2, tasks[0], cfg, opt, mask=cfg.output_masking, is_moe=True)
    snaps.append(("After half 2", *routing_matrix(model, train_set, tasks[0], cfg)))
    if cfg.freeze_experts and probs is not None:
        ids = select_experts_to_freeze(probs.mean(dim=0), cfg.n_experts, free=model.moe.free)
        if ids:
            model.freeze_experts(ids)
    train_phase(model, make_loader(train_set, tasks[1], batch_size=cfg.batch_size), tasks[1], cfg, opt,
                mask=cfg.output_masking, is_moe=True)
    snaps.append(("After task 2", *routing_matrix(model, train_set, tasks[0] + tasks[1], cfg)))
    return _snaps_to_json(snaps)


def revisitation_snapshots(cfg, seed=42, cycles=8, marks=(1, 2, 4, 8)):
    cfg = cfg.with_(routing="learned", model="smoe", epochs_per_phase=1)
    train_set, _, input_dim = load_dataset(cfg.dataset)
    tasks = [list(t) for t in cfg.tasks]
    torch.manual_seed(seed)
    model = SMoE(input_dim, n_output_classes(cfg.tasks), cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    loaders = [make_loader(train_set, t, batch_size=cfg.batch_size) for t in tasks]
    all_classes = [c for t in tasks for c in t]
    snaps = []
    for cycle in range(1, cycles + 1):
        for k, classes in enumerate(tasks):
            train_phase(model, loaders[k], classes, cfg, opt, mask=cfg.output_masking, is_moe=True)
        if cycle in marks:
            snaps.append((f"Cycle {cycle}", *routing_matrix(model, train_set, all_classes, cfg)))
    return _snaps_to_json(snaps)


def _snaps_to_json(snaps):
    return [{"title": t, "matrix": np.asarray(m).tolist(), "valid": np.asarray(v).astype(bool).tolist()}
            for t, m, v in snaps]


def _snaps_from_json(data):
    return [(d["title"], np.array(d["matrix"]), np.array(d["valid"])) for d in data]


def _tick_step(n):
    return 1 if n <= 5 else (2 if n <= 10 else 5)


def draw_allocation(snaps_json, path):
    snaps = _snaps_from_json(snaps_json)
    with plt.rc_context(FIG_RC):
        n_exp = snaps[0][1].shape[1]
        step = _tick_step(n_exp)
        vmax = max(m[v].max() for _, m, v in snaps)
        usage = [(lambda u: u / u.sum() * 100)(m[v].mean(axis=0)) for _, m, v in snaps]
        ymax = max(u.max() for u in usage) * 1.12
        fig = plt.figure(figsize=(TEXT_WIDTH, 3.9))
        gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 0.07], height_ratios=[1.35, 1], wspace=0.38,
                              hspace=0.18, left=0.1, right=0.9, top=0.92, bottom=0.13)
        for col, (title, m, v) in enumerate(snaps):
            ax = fig.add_subplot(gs[0, col])
            im = ax.imshow(m[v], aspect="auto", cmap="viridis", vmin=0, vmax=vmax)
            ax.set_yticks(range(int(v.sum()))); ax.set_yticklabels(np.where(v)[0])
            ax.set_xticks(range(0, n_exp, step)); ax.set_xticklabels([])
            ax.set_title(title, fontweight="bold", pad=4)
            if col == 0:
                ax.set_ylabel("Class")
            axb = fig.add_subplot(gs[1, col])
            axb.bar(range(n_exp), usage[col], color="#4C72B0", width=0.8)
            axb.set_xticks(range(0, n_exp, step)); axb.set_xlim(-0.6, n_exp - 0.4); axb.set_ylim(0, ymax)
            axb.set_xlabel("Expert"); axb.grid(axis="y", linestyle=":", alpha=0.7)
            for side in ("top", "right"):
                axb.spines[side].set_visible(False)
            if col == 0:
                axb.set_ylabel("Share (%)")
            else:
                axb.set_yticklabels([])
        cb = fig.colorbar(im, cax=fig.add_subplot(gs[0, 3]))
        cb.set_label("Routing prob."); cb.locator = plt.MaxNLocator(3); cb.update_ticks()
        plt.savefig(path, dpi=300, facecolor="white")
        plt.show()


def draw_revisitation(snaps_json, path):
    snaps = _snaps_from_json(snaps_json)
    with plt.rc_context(FIG_RC):
        n_exp = snaps[0][1].shape[1]
        step = _tick_step(n_exp)
        vmin = min(m[v].min() for _, m, v in snaps)
        vmax = max(m[v].max() for _, m, v in snaps)
        n = len(snaps)
        fig = plt.figure(figsize=(TEXT_WIDTH, 2.2))
        gs = fig.add_gridspec(1, n + 1, width_ratios=[1] * n + [0.07], wspace=0.22,
                              left=0.08, right=0.9, top=0.87, bottom=0.25)
        for col, (title, m, v) in enumerate(snaps):
            ax = fig.add_subplot(gs[0, col])
            im = ax.imshow(m[v], aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
            classes = np.where(v)[0]
            ax.set_yticks(range(0, len(classes), 2)); ax.set_yticklabels(classes[::2] if col == 0 else [])
            ax.set_xticks(range(0, n_exp, step)); ax.set_xlabel("Expert")
            ax.set_title(title, fontweight="bold", pad=4)
            if col == 0:
                ax.set_ylabel("Class")
        cb = fig.colorbar(im, cax=fig.add_subplot(gs[0, n]))
        cb.set_label("Routing prob."); cb.locator = plt.MaxNLocator(3); cb.update_ticks()
        plt.savefig(path, dpi=300, facecolor="white")
        plt.show()


allocation = cached("routing_allocation", lambda: allocation_snapshots(BASE))
revisitation = cached("routing_revisitation", lambda: revisitation_snapshots(BASE))
draw_allocation(allocation, FIGURES_DIR / "exp2_mnist.png")
draw_revisitation(revisitation, FIGURES_DIR / "heatmap_drift.png")

In [ ]:
def specialization_grid(n_experts=20, n_samples=12, n_shown=5, seed=42):
    # Trains on all ten classes at once (no tasks, no masking) and assigns every test image
    # to its top-1 expert.
    cfg = BASE.with_(n_experts=n_experts, freeze_experts=False, output_masking=False)
    torch.manual_seed(seed)
    train_set, test_set, input_dim = load_dataset("mnist")
    model = SMoE(input_dim, 10, cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    train_phase(model, make_loader(train_set, list(range(10)), batch_size=cfg.batch_size),
                list(range(10)), cfg, opt, mask=False, is_moe=True)
    model.eval()
    top1, targets = [], []
    with torch.no_grad():
        for x, y in DataLoader(test_set, batch_size=500, shuffle=False):
            top1.append(model(x.to(device))[2].argmax(dim=-1).cpu()); targets.append(y)
    top1, targets = torch.cat(top1).numpy(), torch.cat(targets).numpy()
    counts = np.zeros((n_experts, 10), dtype=int)
    for e, c in zip(top1, targets):
        counts[e, c] += 1
    with torch.no_grad():
        correct = sum((model(x.to(device))[0].argmax(1).cpu() == y).sum().item()
                      for x, y in DataLoader(test_set, batch_size=500, shuffle=False))
    shown = [int(e) for e in np.argsort(-counts.sum(axis=1))[:n_shown]]
    raw = test_set.data.numpy() if hasattr(test_set.data, "numpy") else np.asarray(test_set.data)
    grid = [[raw[i].tolist() for i in np.where(top1 == e)[0][:n_samples]] for e in shown]
    return {"counts": counts.tolist(), "shown": shown, "grid": grid,
            "accuracy": correct / len(test_set) * 100}


def draw_grid(data, path):
    with plt.rc_context(FIG_RC):
        L, label_w = TEXT_WIDTH, 0.72
        rows, cols = len(data["grid"]), max(len(g) for g in data["grid"])
        tile = (L - label_w) / cols
        H = tile * rows
        fig = plt.figure(figsize=(L, H))
        for i, (e, imgs) in enumerate(zip(data["shown"], data["grid"])):
            fig.text((label_w - 0.08) / L, 1 - (i + 0.5) / rows, f"Expert {e}", ha="right", va="center")
            for j, img in enumerate(imgs):
                img = np.array(img)
                if img.ndim == 1:
                    side = int(round(np.sqrt(img.size))); img = img.reshape(side, side)
                ax = fig.add_axes([(label_w + j * tile + 0.02) / L, 1 - (i + 1) / rows + 0.02 / H,
                                   (tile - 0.04) / L, (tile - 0.04) / H])
                ax.imshow(img, cmap="gray", vmin=0, vmax=255); ax.axis("off")
        plt.savefig(path, dpi=300, facecolor="white")
        plt.show()


grid = cached("routing_specialization", specialization_grid)
counts = np.array(grid["counts"])
print(f"test accuracy (all ten classes): {grid['accuracy']:.2f}%")
print(f"experts receiving no test image: {int((counts.sum(axis=1) == 0).sum())} of {len(counts)}\n")
print("classes received by each expert (share of its top-1 test images):")
for e in np.argsort(-counts.sum(axis=1)):
    n = counts[e].sum()
    if n:
        top = [c for c in np.argsort(-counts[e])[:3] if counts[e, c]]
        mark = "  <- shown in Figure 4" if e in grid["shown"] else ""
        print(f"  expert {e:>2} (n={n:>4}): " + ", ".join(f"{c}: {counts[e, c] / n * 100:.0f}%" for c in top) + mark)
draw_grid(grid, FIGURES_DIR / "grade_visual_mnist.png")

**Reading the results.** During both halves of Task 1, traffic is spread across all experts without class structure: the balancing loss rewards uniform usage, and nothing in it rewards routing similar images together. Specialization appears only when the model is trained on all classes, where experts group digits by shape. Across revisitation cycles, the routing matrices keep changing: this is routing drift. The shared input projection, which feeds the router and is never frozen, keeps rotating the latent space as each task is trained (Section 5.5 tests this directly).

### 5.3 Why is plasticity reduced under freezing? (Table 1)

Output masking turns off gradients for Task 1 classes while Task 2 is trained, so the network is never taught to lower the Task 1 logits, but plasticity is measured with an unrestricted argmax over all classes. To separate learning the new task from winning that comparison, Task 2 test predictions are computed in two ways:

- **task-incremental accuracy:** argmax restricted to Task 2 classes, which measures only how well the new classes are told apart from each other;
- **class-incremental accuracy:** argmax over all classes, as everywhere else, so the new classes must also outscore the old ones.

The gap between them is the accuracy lost to competition with old classes. Three seeds, with and without freezing, on both datasets.

In [ ]:
def diagnose_two_tasks(cfg, seed):
    torch.manual_seed(seed)
    train_set, _, input_dim = load_dataset(cfg.dataset)
    tasks = [list(t) for t in cfg.tasks]
    n_classes = n_output_classes(cfg.tasks)
    test = test_loaders(cfg.dataset, cfg.tasks)
    t1, t2 = tasks[0], tasks[1]
    model = SMoE(input_dim, n_classes, cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    h1, h2 = make_half_loaders(train_set, t1, batch_size=cfg.batch_size, seed=seed)
    probs = None
    for loader in (h1, h2):
        probs = train_phase(model, loader, t1, cfg, opt, mask=cfg.output_masking, is_moe=True)
    n_frozen = 0
    if cfg.freeze_experts and probs is not None:
        ids = select_experts_to_freeze(probs.mean(dim=0), cfg.n_experts, free=model.moe.free)
        if ids:
            model.freeze_experts(ids)
        n_frozen = len(ids)
    train_phase(model, make_loader(train_set, t2, batch_size=cfg.batch_size), t2, cfg, opt,
                mask=cfg.output_masking, is_moe=True)
    model.eval()
    logits, targets = [], []
    with torch.no_grad():
        for x, y in test[1]:
            logits.append(model(x.to(device))[0].cpu()); targets.append(y)
    logits, targets = torch.cat(logits), torch.cat(targets)
    pred_class = logits.argmax(1)
    old = torch.zeros(logits.size(1), dtype=torch.bool); old[t1] = True
    restricted = logits.clone(); restricted[:, old] = -1e9
    pred_task = restricted.argmax(1)
    return {"seed": seed, "frozen": n_frozen,
            "task_incremental": (pred_task == targets).float().mean().item() * 100,
            "class_incremental": (pred_class == targets).float().mean().item() * 100,
            "share_to_old_classes": torch.isin(pred_class, torch.tensor(t1)).float().mean().item() * 100,
            "mean_logit_old": logits[:, t1].mean().item(), "mean_logit_new": logits[:, t2].mean().item(),
            "confusion": confusion_matrix(targets.numpy(), pred_class.numpy(),
                                          labels=list(range(n_classes))).tolist()}


def run_diagnosis_two_tasks():
    out = {}
    for name, base in [("MNIST", BASE), ("CIFAR-10", CIFAR)]:
        for label, freeze in [("SMoE (freeze)", True), ("SMoE (no freeze)", False)]:
            cfg = base.with_(output_masking=True, freeze_experts=freeze)
            out[f"{name} | {label}"] = [diagnose_two_tasks(cfg, s) for s in DIAGNOSIS_SEEDS]
    return out


diagnosis = cached("diagnosis_two_tasks", run_diagnosis_two_tasks)
print(f"{'':<30} {'task-incr.':>14} {'class-incr.':>14} {'gap':>14} {'to old classes':>15}")
for key, runs in diagnosis.items():
    ti = np.array([r["task_incremental"] for r in runs]); ci = np.array([r["class_incremental"] for r in runs])
    old = np.mean([r["share_to_old_classes"] for r in runs])
    print(f"{key:<30} {ti.mean():>7.1f} ± {ti.std():<4.1f} {ci.mean():>7.1f} ± {ci.std():<4.1f} "
          f"{(ti - ci).mean():>7.1f} ± {(ti - ci).std():<4.1f} {old:>13.1f}%")

print("\nMean logit of old (Task 1) vs new (Task 2) classes on Task 2 test images, CIFAR-10 with freezing:")
for r in diagnosis["CIFAR-10 | SMoE (freeze)"]:
    print(f"  seed {r['seed']}: old {r['mean_logit_old']:+.2f} | new {r['mean_logit_new']:+.2f} | "
          f"difference {r['mean_logit_old'] - r['mean_logit_new']:+.2f}")

In [ ]:
def show_confusion(key, seed_index=0):
    # Confusion matrix of Task 2 test images (rows: true class; columns: predicted class).
    base = CIFAR if key.startswith("CIFAR") else BASE
    t1, t2 = list(base.tasks[0]), list(base.tasks[1])
    cm = np.array(diagnosis[key][seed_index]["confusion"])
    print(f"\n{key}, seed {DIAGNOSIS_SEEDS[seed_index]}")
    print("          " + "".join(f"{c:>6}" for c in range(cm.shape[1])))
    print("          " + "".join(f"{'(T1)' if c in t1 else '(T2)':>6}" for c in range(cm.shape[1])))
    for c in t2:
        row = cm[c]
        print(f"  true {c}:" + "".join(f"{v:>6}" for v in row) +
              f"   -> correct {row[c] / row.sum() * 100:4.1f}%, to Task 1 {row[t1].sum() / row.sum() * 100:4.1f}%")

for key in diagnosis:
    show_confusion(key)

**Reading the results.** Task-incremental accuracy is nearly the same with and without freezing: freezing barely changes how well Task 2 is learned. The difference lies in the class-incremental decision, where Task 1 classes, whose logits remain high, win on Task 2 images. The confusion matrices show the errors concentrating in specific Task 1 classes rather than spreading among Task 2 classes. Without freezing, the gap nearly vanishes on MNIST: the network moves its representation away from Task 1 and the old logits decay.

### 5.4 Influence of morphological characteristics on routing (Table 2)

Since the router groups digits by shape, modularity could help more when each task contains morphologically similar digits. The MNIST protocol is extended to three tasks grouped by shape (`MORPHOLOGICAL_MNIST`), with freezing re-applied at each transition. To tell whether the grouping itself matters, the same models are trained on a control with the same task sizes but numerical order (`CONTROL_MNIST`), which mixes shapes within each task. The question is whether the SMoE's advantage over the masked MLP is larger with the morphological grouping than with the control.

In [ ]:
MORPH = BASE.with_(tasks=MORPHOLOGICAL_MNIST)
CONTROL = BASE.with_(tasks=CONTROL_MNIST)

def three_task_configurations(base, include_plain_mlp):
    configs = [("MLP", base.with_(model="mlp", output_masking=False))] if include_plain_mlp else []
    return configs + [("MLP (masked)",        base.with_(model="mlp", output_masking=True)),
                      ("SMoE (no freeze)",    base.with_(output_masking=True, freeze_experts=False)),
                      ("SMoE (freeze)",       base.with_(output_masking=True, freeze_experts=True)),
                      ("SMoE (ideal router)", base.with_(routing="oracle"))]

three_morph = cached("three_tasks_morphological",
                     lambda: [run_config(c, l) for l, c in three_task_configurations(MORPH, True)])
three_control = cached("three_tasks_control",
                       lambda: [run_config(c, l) for l, c in three_task_configurations(CONTROL, False)])

for r in three_morph:
    print_matrix(r)
print_summary(three_morph, "three tasks, morphological grouping")
print_summary(three_control, "three tasks, control (numerical order)")

print("\nTable 2 (T1 initial | T1 final | T2 final | T3), LaTeX rows:")
for r in three_morph[1:]:
    s = summarize(r); m, d = s["matrix_mean"], s["matrix_std"]
    print(f"\\textbf{{{s['label']}}} & " + " & ".join(
        f"{m[i][j]:.2f} $\\pm$ {d[i][j]:.2f}" for i, j in [(0, 0), (2, 0), (2, 1), (2, 2)]) + " \\\\")

print("\nAdvantage of each SMoE variant over the masked MLP (retention, percentage points):")
morph_s = {summarize(r)["label"]: summarize(r) for r in three_morph}
ctrl_s = {summarize(r)["label"]: summarize(r) for r in three_control}
for name in ["SMoE (no freeze)", "SMoE (freeze)", "SMoE (ideal router)"]:
    gm = morph_s[name]["retention"] - morph_s["MLP (masked)"]["retention"]
    gc = ctrl_s[name]["retention"] - ctrl_s["MLP (masked)"]["retention"]
    print(f"  {name:<22} morphological {gm:+7.2f} | control {gc:+7.2f} | effect of the grouping {gm - gc:+7.2f}")

In [ ]:
def diagnose_three_tasks(cfg, seed, targets=(0, 1)):
    # Trains the full three-task sequence once and evaluates each earlier task in two ways:
    # restricted to its own classes (task-incremental) and over all classes (class-incremental).
    torch.manual_seed(seed)
    train_set, _, input_dim = load_dataset(cfg.dataset)
    tasks = [list(t) for t in cfg.tasks]
    n_classes = n_output_classes(cfg.tasks)
    test = test_loaders(cfg.dataset, cfg.tasks)
    mask_first = cfg.output_masking if cfg.mask_first_task is None else cfg.mask_first_task
    model = SMoE(input_dim, n_classes, cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    frozen = []
    for i, classes in enumerate(tasks):
        if i == 0 and cfg.split_first_task:
            phases = list(make_half_loaders(train_set, classes, batch_size=cfg.batch_size, seed=seed))
        else:
            phases = [make_loader(train_set, classes, batch_size=cfg.batch_size)]
        mask = mask_first if i == 0 else cfg.output_masking
        probs = None
        for loader in phases:
            probs = train_phase(model, loader, classes, cfg, opt, mask=mask, is_moe=True)
        if i < len(tasks) - 1 and cfg.freeze_experts and probs is not None:
            ids = select_experts_to_freeze(probs.mean(dim=0), cfg.n_experts, free=model.moe.free,
                                           min_reserve=reserve_size(cfg, len(tasks) - (i + 1)))
            if ids:
                model.freeze_experts(ids)
            frozen.append(len(ids))
    out = {"seed": seed, "frozen": frozen, "tasks": {}}
    for t in targets:
        own = tasks[t]
        model.eval()
        logits, ys = [], []
        with torch.no_grad():
            for x, y in test[t]:
                logits.append(model(x.to(device))[0].cpu()); ys.append(y)
        logits, ys = torch.cat(logits), torch.cat(ys)
        pred = logits.argmax(1)
        outside = torch.ones(logits.size(1), dtype=torch.bool); outside[own] = False
        restricted = logits.clone(); restricted[:, outside] = -1e9
        out["tasks"][str(t)] = {      # string keys: identical whether computed or loaded from JSON
            "task_incremental": (restricted.argmax(1) == ys).float().mean().item() * 100,
            "class_incremental": (pred == ys).float().mean().item() * 100,
            "share_to_task": [torch.isin(pred, torch.tensor(c)).float().mean().item() * 100 for c in tasks]}
    return out


def run_diagnosis_three_tasks():
    return {label: [diagnose_three_tasks(MORPH.with_(output_masking=True, freeze_experts=f), s)
                    for s in DIAGNOSIS_SEEDS]
            for label, f in [("SMoE (freeze)", True), ("SMoE (no freeze)", False)]}


diagnosis3 = cached("diagnosis_three_tasks", run_diagnosis_three_tasks)
for label, runs in diagnosis3.items():
    print(f"\n{label}")
    for t in ("0", "1"):
        ti = [r["tasks"][t]["task_incremental"] for r in runs]
        ci = [r["tasks"][t]["class_incremental"] for r in runs]
        dest = np.array([r["tasks"][t]["share_to_task"] for r in runs])
        print(f"  Task {int(t) + 1}: task-incr. {min(ti):.1f}-{max(ti):.1f}% | class-incr. {min(ci):.1f}-{max(ci):.1f}% | "
              "predictions per task (min-max): " +
              ", ".join(f"T{k + 1} {dest[:, k].min():.0f}-{dest[:, k].max():.0f}%" for k in range(dest.shape[1])))

reserve = cached("three_tasks_reserve", lambda: [run_config(
    MORPH.with_(output_masking=True, freeze_experts=True, reserve_policy="proportional"),
    "SMoE (freeze, proportional reserve)")])
print_summary([r for r in three_morph if r["label"] == "SMoE (freeze)"] + reserve,
              "capacity reservation: minimum (one free expert) vs proportional")

**Reading the results.** Forgetting compounds with three tasks, through two distinct mechanisms. Task 1 collapses to near chance even when evaluated only among its own classes, with and without freezing, so capacity is not the cause: the drift of the shared projection accumulates over two transitions. Task 2 is still learned in isolation but loses the class-incremental decision to Task 3, the most recently trained task (recency bias). Reserving capacity for later tasks does not help either mechanism. Grouping by shape itself makes no measurable difference: against the control configuration, retention changes by at most 4.8 points for any model, within one standard deviation, and the gap between each SMoE variant and the masked MLP shifts by less than 4 points, in both directions.

### 5.5 Is the shared projection responsible for the drift?

Sections 5.2 and 5.4 attribute the drift to the shared input projection, the only component that is never frozen. Here the projection is removed, so the router and experts operate on the flattened image. Because each expert is sized relative to its input, removing the projection would multiply the number of parameters (by 32 on MNIST); the experts' hidden width is therefore reduced so that the models with and without the projection have the same number of parameters. The comparison uses the results with the projection from Sections 5.1 and 5.4.

In [ ]:
H_EQ = equalized_expert_hidden(BASE, BASE.with_(use_input_projection=False))
NO_PROJ = dict(use_input_projection=False, expert_hidden=H_EQ)

def instantiated_parameters(cfg):
    _, _, input_dim = load_dataset(cfg.dataset)
    if cfg.routing == "oracle":
        m = OracleSMoE(input_dim, cfg, device); m.add_classes(list(range(10)))
    else:
        m = SMoE(input_dim, 10, cfg)
    return sum(p.numel() for p in m.parameters())

print(f"expert hidden width without the projection: {H_EQ}")
for label, cfg in [("SMoE, with projection", BASE), ("SMoE, without (equalized)", BASE.with_(**NO_PROJ)),
                   ("ideal router, with projection", BASE.with_(routing="oracle")),
                   ("ideal router, without (equalized)", BASE.with_(routing="oracle", **NO_PROJ))]:
    print(f"  {label:<36} {instantiated_parameters(cfg):>10,} parameters")

def projection_configurations(base):
    return [("SMoE (no freeze), no projection",    base.with_(output_masking=True, freeze_experts=False, **NO_PROJ)),
            ("SMoE (freeze), no projection",       base.with_(output_masking=True, freeze_experts=True, **NO_PROJ)),
            ("SMoE (ideal router), no projection", base.with_(routing="oracle", **NO_PROJ))]

proj_two = cached("projection_two_tasks", lambda: [run_config(c, l) for l, c in projection_configurations(BASE)])
proj_three = cached("projection_three_tasks", lambda: [run_config(c, l) for l, c in projection_configurations(MORPH)])

with_two = {summarize(r)["label"]: summarize(r) for r in main_mnist}
with_three = {summarize(r)["label"]: summarize(r) for r in three_morph}
print("\nTwo tasks (retention / plasticity): with -> without the projection")
for r in proj_two:
    s = summarize(r); w = with_two[s["label"].replace(", no projection", "")]
    print(f"  {s['label']:<36} {w['retention']:6.2f} -> {s['retention']:6.2f} | {w['plasticity']:6.2f} -> {s['plasticity']:6.2f}")
print("\nThree tasks (T1 final / T2 final / T3): with -> without the projection")
for r in proj_three:
    s = summarize(r); w = with_three[s["label"].replace(", no projection", "")]
    a, b = w["matrix_mean"], s["matrix_mean"]
    print(f"  {s['label']:<36} " + " | ".join(f"{a[2][j]:6.2f} -> {b[2][j]:6.2f}" for j in range(3)))

**Reading the results.** At the same number of parameters, removing the projection raises retention by 31 to 33 points in every SMoE variant with two tasks, and by 23 to 33 points with three; frozen experts then keep 79% of Task 1 instead of 5%. Without the projection, the unfrozen SMoE reaches the masked MLP, so the disadvantage of modularity in Section 5.1 depends on the shared projection. The gain costs plasticity: the experts operate directly on the image through a narrower hidden layer, and learn new tasks less well, most clearly when freezing leaves a single expert free.

### 5.6 Additional analyses

Variations of the reference configuration (SMoE with freezing, MNIST, two tasks), each changing a single field: the number of experts, $k$, the balancing weight $\alpha$, and whether output masking is applied from the first task or only from the second onward. They use three seeds. The allocation and revisitation analyses of Section 5.2 are also repeated with 4 and 20 experts.

In [ ]:
ABLATION_BASE = BASE.with_(seeds=(42, 123, 2024))
ABLATIONS = {"n_experts": [4, 10, 20], "top_k": [1, 2, 4], "alpha": [0.1, 1.0, 10.0], "mask_first_task": [None, False]}

ablations = cached("ablations", lambda: {
    field: [run_config(ABLATION_BASE.with_(**{field: v}), f"{field}={v}") for v in values]
    for field, values in ABLATIONS.items()})

for field, results in ablations.items():
    print_summary(results, f"varying {field}")

for n in (4, 20):
    draw_allocation(cached(f"routing_allocation_{n}_experts", lambda: allocation_snapshots(BASE.with_(n_experts=n))),
                    FIGURES_DIR / f"exp2_mnist_{n}_experts.png")
    draw_revisitation(cached(f"routing_revisitation_{n}_experts", lambda: revisitation_snapshots(BASE.with_(n_experts=n))),
                      FIGURES_DIR / f"heatmap_drift_{n}_experts.png")

**Reading the results.** More experts do not isolate tasks better: retention decreases from 4 to 20 experts, and plasticity becomes unstable with 20, when freezing leaves a single expert out of twenty free. $k = 2$ retains as much as $k = 4$ and more than $k = 1$, with more plasticity than $k = 4$. $\alpha$ changes retention by less than its spread across seeds, and $\alpha = 10$ gives the most stable plasticity. Masking only from the second task onward raises retention by about 8 points but lowers plasticity by about 39: the first task then pushes down the logits of future classes, while later tasks never push down those of earlier ones. This is why masking is applied in every task.

## 6. Figure 1b: experimental protocol

The protocol diagram, drawn at its printed size next to Figure 1a.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_protocol(path=FIGURES_DIR / "protocol.png"):
    W, H = 2.36, 2.14      # inches: 0.40 of the text width, the height of Figure 1a
    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10}):
        fig = plt.figure(figsize=(W, H))
        ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")

        def box(cx, cy, w, h, text, fc="#EAF1F8", ec="#4C72B0"):
            ax.add_patch(FancyBboxPatch((cx - w / 2, cy - h / 2), w, h, boxstyle="round,pad=0.0,rounding_size=0.04",
                                        linewidth=1.0, edgecolor=ec, facecolor=fc))
            ax.text(cx, cy, text, ha="center", va="center")

        def arrow(x0, y0, x1, y1):
            ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=8,
                                         linewidth=1.0, color="#2C3E50"))

        h, cx = 0.30, W / 2
        ys = [H - 0.20, H - 0.64, H - 1.08, H - 1.52, H - 1.96]
        xl, xr = W / 2 - 0.57, W / 2 + 0.57
        box(cx, ys[0], 2.26, h, "Task 1, half 1 (0–4)")
        box(cx, ys[1], 2.26, h, "Task 1, half 2 (0–4)")
        box(cx, ys[2], 2.26, h, "Task 2 (5–9)")
        box(xl, ys[3], 1.08, h, "Test 0–4"); box(xr, ys[3], 1.08, h, "Test 5–9")
        box(xl, ys[4], 1.08, h, "Forgetting", "#FCEEE3", "#DD8452")
        box(xr, ys[4], 1.08, h, "Plasticity", "#FCEEE3", "#DD8452")
        arrow(cx, ys[0] - h / 2, cx, ys[1] + h / 2); arrow(cx, ys[1] - h / 2, cx, ys[2] + h / 2)
        arrow(cx - 0.25, ys[2] - h / 2, xl, ys[3] + h / 2); arrow(cx + 0.25, ys[2] - h / 2, xr, ys[3] + h / 2)
        arrow(xl, ys[3] - h / 2, xl, ys[4] + h / 2); arrow(xr, ys[3] - h / 2, xr, ys[4] + h / 2)
        plt.savefig(path, dpi=300, facecolor="white")
        plt.show()

draw_protocol()

## 7. Citation

If you use this code, please cite:

```bibtex
@inproceedings{lopes2026smoe,
  title     = {Can Sparse Mixture-of-Experts Modularity Mitigate Catastrophic Forgetting in Class-Incremental Learning?},
  author    = {Lopes, Arthur Estefanato and Mutz, Filipe},
  booktitle = {[VENUE PLACEHOLDER]},
  year      = {2026}
}
```

Repository: [REPOSITORY URL PLACEHOLDER]